In [8]:
from pathlib import Path
import numpy as np
import sys
sys.path.append(str(Path("../src").resolve()))# to look inside src folder so it can import files
import pandas as pd

from config import (
    RAW_ROOT,
    PROCESSED_GROUP1,
    NOTCH_FREQ,
    LOW_FREQ,
    HIGH_FREQ,
    TARGET_SFREQ,
    WINDOW_SIZE_SEC,
    STEP_SEC,
    BAD_CHANNEL_Z_THRESH
)

from utils import (
    load_raw_edf,
    prepare_raw,
    apply_filters,
    parse_chbmit_summary,
    detect_bad_channels,
    reorder_channels,
    zscore_normalize,
    create_windows_and_labels,
    save_processed_data,
   
)


# File setup for one patient

patient_name = "chb01"
patient_folder = RAW_ROOT / patient_name
summary_path = patient_folder / f"{patient_name}-summary.txt"

# output folder for this patient
output_folder = PROCESSED_GROUP1 / patient_name
output_folder.mkdir(parents=True, exist_ok=True) # this is to creat the folder if doesnt exist and dont crash if its not there 

# get all EDF files in this patient folder 
#making sure that it is sorted 
edf_files = sorted(patient_folder.glob("*.edf"))

print(f"Processing patient: {patient_name}")
print(f"Found {len(edf_files)} EDF files")


#  Read seizure information from summary file once

summary_dict = parse_chbmit_summary(summary_path)

#  Build expected channel order from first file
#so then it could reordered every other file  to mathc the same channel order 
#load the first channel names 
first_raw = load_raw_edf(edf_files[0])
first_raw = prepare_raw(first_raw)
#saving the names of the channel 
expected_channels = first_raw.ch_names.copy()
#checking them
print("\nExpected channels from first file:")
print(expected_channels)

# Process each EDF file in a loop, applying the same steps to each one
summary_rows = [] # to store summary info for each file for the final summary CSV

for file_path in edf_files:
    file_name = file_path.name
    seizure_intervals = summary_dict.get(file_name, [])

    print(f"Processing file: {file_name}")
    print(f"Seizure intervals: {seizure_intervals}")

    # Load raw EDF file
  
    raw = load_raw_edf(file_path)
    print("Loaded raw file successfully")

 
    # Keep EEG channels only and set them as EEG type
  
    raw = prepare_raw(raw)

    print("EEG channels kept:")
    print(raw.ch_names)
    print(f"Number of EEG channels: {len(raw.ch_names)}")

  
    # Reorder channels to ensure consistency
  
    raw = reorder_channels(raw, expected_channels)

    # save channel names if needed
    ch_names = raw.ch_names.copy()


    # Check sampling rate
  
    print(f"Original sampling rate: {raw.info['sfreq']} Hz")


    # Apply filtering
  
    raw_before_filter = raw.copy()

    raw = apply_filters(
        raw,
        notch_freq=NOTCH_FREQ,
        l_freq=LOW_FREQ,
        h_freq=HIGH_FREQ
    )

    raw_after_filter = raw.copy()

    print(f"Filtering done: notch {NOTCH_FREQ} Hz + band-pass {LOW_FREQ}-{HIGH_FREQ} Hz")

    # Detect suspicious bad channels

    bad_channels, channel_std = detect_bad_channels(raw, z_thresh=BAD_CHANNEL_Z_THRESH)
    raw.info["bads"] = bad_channels

    print(f"Suspicious bad channels: {bad_channels}")

    # Convert EEG signal to NumPy array
   
    data_before_norm = raw.get_data()
    sfreq = raw.info["sfreq"]

    print(f"Data shape before normalization: {data_before_norm.shape}")
    print(f"Sampling frequency used: {sfreq}")

 
    # Normalize

    data_after_norm = zscore_normalize(data_before_norm.copy())
    data = data_after_norm

    print("Normalization done")


    # Split into windows and label them

    X, y = create_windows_and_labels(
        data=data,
        sfreq=sfreq,
        window_size_sec=WINDOW_SIZE_SEC,
        step_sec=STEP_SEC,
        seizure_intervals=seizure_intervals
    )

    print(f"Windowed data shape: {X.shape}")
    print(f"Labels shape: {y.shape}")
    print(f"Number of seizure windows: {np.sum(y)}")
    print(f"Number of non-seizure windows: {len(y) - np.sum(y)}")

    
    # Save processed result for this file
    
 
    save_path = output_folder / f"{Path(file_name).stem}_processed.npz"
    save_processed_data(save_path, X, y)

    print(f"Preprocessing finished successfully for {file_name}")
      #this is for the summry to loop throgh all files
    row = {
        "file_name": file_name,
        "n_channels": len(raw.ch_names),
        "sampling_rate": raw.info["sfreq"],
        "duration_sec": raw.n_times / raw.info["sfreq"],
        "n_seizure_intervals": len(seizure_intervals),
        "n_bad_channels": len(bad_channels),
        "n_windows": len(y),
        "n_seizure_windows": int(np.sum(y == 1)),
        "n_nonseizure_windows": int(np.sum(y == 0))
    }

    summary_rows.append(row)
print("\nAll files in chb01 processed successfully.")

# Save summary AFTER the loop
df_summary = pd.DataFrame(summary_rows)

summary_save_path = output_folder / f"{patient_name}_processing_summary.csv"
df_summary.to_csv(summary_save_path, index=False)

print("\nSummary saved to:", summary_save_path)
print("\nPreview:")
print(df_summary.head())

Processing patient: chb01
Found 42 EDF files


C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)



Expected channels from first file:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Processing file: chb01_01.edf
Seizure intervals: []


C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Loaded raw file successfully
EEG channels kept:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels: 23
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000070
F7-T7: 0.000042
T7-P7: 0.000025
P7-O1: 0.000018
FP1-F3: 0.000069
F3-C3: 0.000026
C3-P3: 0.000019
P3-O1: 0.000022
FP2-F4: 0.000058
F4-C4: 0.000023
C4-P4: 0.000018
P4-O2: 0.000022
FP2-F8: 0.000058
F8-T8: 0.000037
T8-P8-0: 0.000026
P8-O2: 0.000026
FZ-CZ: 0.000022
CZ-PZ: 0.000021
P7-T7: 0.000025
T7-FT9: 0.000029
FT9-FT10: 0.000053
FT10-T8: 0.000028
T8-P8-1: 0.000026

Suspicious channels: ['FP1-F7', 'FP1-F3']
Suspicious bad channels: ['FP1-F7', 'FP1-F3']
Data shape before normalization: (23, 921600)
Sampling frequency used: 256.0
Normalization done
Windowed data shape: (1439, 23, 

...

Loaded raw file successfully
EEG channels kept:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels: 23
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000055
F7-T7: 0.000030
T7-P7: 0.000032
P7-O1: 0.000035
FP1-F3: 0.000057
F3-C3: 0.000049
C3-P3: 0.000056
P3-O1: 0.000053
FP2-F4: 0.000080
F4-C4: 0.000075
C4-P4: 0.000049
P4-O2: 0.000050
FP2-F8: 0.000044
F8-T8: 0.000031
T8-P8-0: 0.000042
P8-O2: 0.000056
FZ-CZ: 0.000076
CZ-PZ: 0.000095
P7-T7: 0.000032
T7-FT9: 0.000023
FT9-FT10: 0.000042
FT10-T8: 0.000025
T8-P8-1: 0.000042

Suspicious channels: ['CZ-PZ']
Suspicious bad channels: ['CZ-PZ']
Data shape before normalization: (23, 921600)
Sampling frequency used: 256.0
Normalization done
Windowed data shape: (1439, 23, 1280)
Labels shape: (1

C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Loaded raw file successfully
EEG channels kept:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels: 23
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000031
F7-T7: 0.000025
T7-P7: 0.000025
P7-O1: 0.000023
FP1-F3: 0.000035
F3-C3: 0.000032
C3-P3: 0.000023
P3-O1: 0.000028
FP2-F4: 0.000036
F4-C4: 0.000034
C4-P4: 0.000023
P4-O2: 0.000034
FP2-F8: 0.000031
F8-T8: 0.000028
T8-P8-0: 0.000032
P8-O2: 0.000035
FZ-CZ: 0.000042
CZ-PZ: 0.000037
P7-T7: 0.000025
T7-FT9: 0.000019
FT9-FT10: 0.000038
FT10-T8: 0.000021
T8-P8-1: 0.000032

Suspicious channels: []
Suspicious bad channels: []
Data shape before normalization: (23, 921600)
Sampling frequency used: 256.0
Normalization done
Windowed data shape: (1439, 23, 1280)
Labels shape: (1439,)
Number o

C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Loaded raw file successfully
EEG channels kept:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels: 23
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000069
F7-T7: 0.000050
T7-P7: 0.000030
P7-O1: 0.000020
FP1-F3: 0.000063
F3-C3: 0.000028
C3-P3: 0.000024
P3-O1: 0.000023
FP2-F4: 0.000057
F4-C4: 0.000034
C4-P4: 0.000021
P4-O2: 0.000022
FP2-F8: 0.000057
F8-T8: 0.000038
T8-P8-0: 0.000036
P8-O2: 0.000026
FZ-CZ: 0.000048
CZ-PZ: 0.000052
P7-T7: 0.000030
T7-FT9: 0.000038
FT9-FT10: 0.000065
FT10-T8: 0.000039
T8-P8-1: 0.000036

Suspicious channels: []
Suspicious bad channels: []
Data shape before normalization: (23, 921600)
Sampling frequency used: 256.0
Normalization done
Windowed data shape: (1439, 23, 1280)
Labels shape: (1439,)
Number o

In [1]:
#chedking all usable patients against that channel of 21 set 
from pathlib import Path
import pandas as pd
import sys

sys.path.append(str(Path("../src").resolve()))

from config import RAW_ROOT
from utils import load_raw_edf, prepare_raw

# your usable patients
usable_patients = [
    "chb01","chb02","chb03","chb04","chb05","chb06","chb07","chb08",
    "chb09","chb10","chb11","chb13","chb14","chb15","chb16","chb17",
    "chb19","chb20","chb21","chb22","chb23","chb24"
]

# candidate common 21-channel set
COMMON_21_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ", "P7-T7", "T7-FT9",
    "FT9-FT10"
]

rows = []

for patient_name in usable_patients:
    patient_folder = RAW_ROOT / patient_name
    edf_files = sorted(patient_folder.glob("*.edf"))

    patient_ok = True
    missing_summary = set()

    for file_path in edf_files:
        try:
            raw = load_raw_edf(file_path)
            raw = prepare_raw(raw)

            current_channels = set(raw.ch_names)
            missing_channels = [ch for ch in COMMON_21_CHANNELS if ch not in current_channels]

            if len(missing_channels) > 0:
                patient_ok = False
                missing_summary.update(missing_channels)

        except Exception as e:
            patient_ok = False
            missing_summary.add(f"ERROR: {str(e)}")

    rows.append({
        "patient_name": patient_name,
        "fits_21_channel_set": patient_ok,
        "n_missing_unique_items": len(missing_summary),
        "missing_items": ", ".join(sorted(missing_summary))
    })

df_check = pd.DataFrame(rows)

print(df_check)
df_check.to_csv("usable_patients_21_channel_check.csv", index=False)

print("\nSaved: usable_patients_21_channel_check.csv")

C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Ch

   patient_name  fits_21_channel_set  n_missing_unique_items  \
0         chb01                False                       1   
1         chb02                False                       1   
2         chb03                False                       1   
3         chb04                False                       1   
4         chb05                False                       1   
5         chb06                False                       1   
6         chb07                False                       1   
7         chb08                False                       1   
8         chb09                False                       1   
9         chb10                False                       1   
10        chb11                False                       1   
11        chb13                False                       4   
12        chb14                False                       1   
13        chb15                False                       4   
14        chb16                False    